# 🔥 K-water Co-Scientist FT 학습 v3 — 끊겨도 이어지는 학습

**KONI-Llama3-8B / Qwen2.5-7B QLoRA** + 관제센터 실시간 지표 + **끊김 완전 대응**.

| v3 신기능 | 설명 |
|---|---|
| ↻ 지표 이어쓰기 | 재실행 시 기존 metrics.jsonl을 이어받아 누적 (resume 이벤트 기록) |
| 🗂️ run 버전관리 | `ft_logs/index.json` 자동 유지 → 관제센터 이력 테이블 |
| 💾 Drive 체크포인트 | 100스텝마다 저장, 재실행 시 **마지막 지점부터 자동 재개** |
| 🔌 로컬 미러 | GitHub 장애여도 지표는 Drive에 계속 기록 |

### 사용 순서
1. 🗝️ Secrets: `GITHUB_TOKEN` + **이 노트북 액세스 토글 ON** (노트북마다!)
2. 📁 학습데이터 jsonl 업로드 → `DATA_PATH` 확인
3. **T4 + `SMOKE=True`** 검증 → **L4 + `SMOKE=False`** 본 학습
4. 위에서부터 전부 실행

### 🔌 끊기지 않게 / 끊겨도 되게 (Pro+ 기준)
- **백그라운드 실행**: 런타임 → 런타임 유형 변경 → *백그라운드 실행* 체크(Pro+ 전용) → 실행 후 브라우저를 닫아도 계속 (최대 24h)
- **그래도 끊기면**: 같은 노트북 다시 열고 `RUN_ID` 그대로 전체 재실행 → ① 지표 이어받고 ② 마지막 체크포인트에서 학습이 자동 재개됩니다. 유실 최대 100스텝.
- 세션 한계(24h)는 정책상 존재 → 체크포인트+재개가 근본 대책 (run_001 규모는 한 세션에 충분)

> ⚠️ 학습데이터는 public repo에 push 금지 — 세션 업로드/Drive만.

In [ ]:
#@title 1️⃣ 설정
RUN_ID      = "run_001"           # 재실행 시 같은 값 유지하면 자동 이어짐
SMOKE       = True                # ✅ 처음 True(20스텝) → 통과 후 False
MODEL_ID    = "KISTI-KONI/KONI-Llama3-8B-Instruct-20240729"  # HF 최신 버전 확인. 대안: "Qwen/Qwen2.5-7B-Instruct"
DATA_PATH   = "/content/ft_dataset.jsonl"
REPO        = "newcave/water-tech-agent"
MOUNT_DRIVE = True                # 체크포인트·지표 미러를 Drive에 (강력 권장)
EPOCHS      = 2
LR          = 2e-4
LOG_EVERY   = 10
SAVE_EVERY  = 100                 # 체크포인트 주기 (스텝)
EVAL_RATIO  = 0.03

In [ ]:
#@title 2️⃣ GPU 자동 프리셋 + 저장 위치 (Drive 권장)
import os, torch
gpu = torch.cuda.get_device_name(0)
if "A100" in gpu:
    P = dict(dtype=torch.bfloat16, bf16=True,  fp16=False, bs=4, accum=4,  seqlen=2048)
elif "L4" in gpu:
    P = dict(dtype=torch.bfloat16, bf16=True,  fp16=False, bs=2, accum=8,  seqlen=2048)
else:  # T4 등 (bf16 미지원)
    P = dict(dtype=torch.float16,  bf16=False, fp16=True,  bs=1, accum=16, seqlen=1024)
print(f"🖥️ {gpu}\n   {P}  (유효배치 {P['bs']*P['accum']})")

BASE = f"/content/kwater_ft/{RUN_ID}"
if MOUNT_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        BASE = f"/content/drive/MyDrive/kwater_ft/{RUN_ID}"
    except Exception as e:
        print("⚠️ Drive 마운트 생략 →", e)
CKPT_DIR = f"{BASE}/checkpoints"
os.makedirs(CKPT_DIR, exist_ok=True)
print("💾 저장 위치:", BASE)

In [ ]:
#@title 3️⃣ 라이브러리 설치 (~2분)
!pip install -q -U bitsandbytes peft accelerate datasets
import transformers, peft, bitsandbytes
print("transformers", transformers.__version__, "| peft", peft.__version__, "| bnb", bitsandbytes.__version__)

In [ ]:
#@title 4️⃣ 관제센터 로거 v3 (이어쓰기 + index.json + 로컬미러)
import json, time, base64, requests
from pathlib import Path
from google.colab import userdata

class MissionControlLogger:
    """ft_logs/<RUN_ID>/metrics.jsonl + ft_logs/index.json 유지.
    - 재실행 시 기존 지표 이어받음(resume) → 끊겨도 기록 누적
    - 모든 지표를 로컬(BASE)에도 미러 → GitHub 장애 대비
    - push 실패는 경고만, 학습은 절대 중단 안 됨"""

    def __init__(self, repo, run_id, local_base):
        self.run_id = run_id
        self.api = f"https://api.github.com/repos/{repo}/contents"
        self.h = {"Authorization": f"Bearer {userdata.get('GITHUB_TOKEN')}"}
        self.local = Path(local_base); self.local.mkdir(parents=True, exist_ok=True)
        self.lines, self._n, self.resumed = [], 0, False
        old = self._get(f"ft_logs/{run_id}/metrics.jsonl")           # ↻ 이어받기
        if old:
            self.lines = [l for l in old.splitlines() if l.strip()]
            self.resumed = True
            self.log({"event": "resume", "ts": int(time.time())}, push=True)
            print(f"↻ 기존 지표 {len(self.lines)}줄 이어받음 (resume)")

    def _get(self, path):
        try:
            r = requests.get(f"{self.api}/{path}", headers=self.h, timeout=10)
            if r.status_code == 200:
                return base64.b64decode(r.json()["content"]).decode()
        except Exception:
            pass
        return None

    def _put(self, path, text, msg):
        try:
            r = requests.get(f"{self.api}/{path}", headers=self.h, timeout=10)
            sha = r.json().get("sha") if r.status_code == 200 else None
            body = {"message": msg, "content": base64.b64encode(text.encode()).decode()}
            if sha:
                body["sha"] = sha
            r = requests.put(f"{self.api}/{path}", headers=self.h, json=body, timeout=15)
            if r.status_code not in (200, 201):
                print(f"⚠️ push {r.status_code}: {r.text[:100]}")
        except Exception as e:
            print("⚠️ push 실패(무시):", e)

    def log(self, d, push=True):
        self.lines.append(json.dumps(d, ensure_ascii=False))
        try:                                                          # 로컬 미러 (항상)
            (self.local / "metrics.jsonl").write_text("\n".join(self.lines) + "\n", encoding="utf-8")
        except Exception:
            pass
        if push:
            self.push(event=d.get("event"))

    def push(self, event=None):
        self._put(f"ft_logs/{self.run_id}/metrics.jsonl",
                  "\n".join(self.lines) + "\n", f"ft {self.run_id} {time.strftime('%H:%M:%S')}")
        self._n += 1
        if event in ("start", "end", "resume") or self._n % 5 == 0:   # index는 5회마다
            self._update_index(done=(event == "end"))

    def _update_index(self, done=False):
        steps, fl, started, model, g = 0, None, None, None, None
        for ln in self.lines:
            try:
                d = json.loads(ln)
            except Exception:
                continue
            if d.get("event") == "start":
                started, model, g = d.get("ts"), d.get("model"), d.get("gpu")
            elif "step" in d:
                steps = max(steps, d.get("step") or 0)
                if d.get("loss") is not None:
                    fl = d["loss"]
        entry = dict(id=self.run_id, status="✅ 완료" if done else "🟢 학습 중",
                     model=model, gpu=g, steps=steps,
                     final_loss=round(fl, 4) if fl is not None else "-",
                     started=started, last_ts=int(time.time()))
        idx = {"runs": []}
        old = self._get("ft_logs/index.json")
        if old:
            try:
                idx = json.loads(old)
            except Exception:
                pass
        idx["runs"] = [r for r in idx.get("runs", []) if r.get("id") != self.run_id] + [entry]
        self._put("ft_logs/index.json", json.dumps(idx, ensure_ascii=False, indent=1),
                  f"index {self.run_id}")

logger = MissionControlLogger(REPO, RUN_ID, BASE)
print("로거 준비 완료 → ft_logs/" + RUN_ID)

In [ ]:
#@title 5️⃣ 학습데이터 로드 (형식 자동 감지)
import json as _json, os
assert os.path.exists(DATA_PATH), f"❌ {DATA_PATH} 없음 — 좌측 📁 패널에 jsonl 업로드 후 경로 확인"

raw = [_json.loads(l) for l in open(DATA_PATH, encoding="utf-8") if l.strip()]

def to_pair(r):
    if "messages" in r:
        msgs = r["messages"]
        user = "\n".join(m["content"] for m in msgs if m["role"] in ("user", "system"))
        asst = "\n".join(m["content"] for m in msgs if m["role"] == "assistant")
        return user, asst
    for ki, ko in [("instruction", "output"), ("prompt", "completion"),
                   ("input", "output"), ("question", "answer")]:
        if ki in r and ko in r:
            u = r[ki]
            if ki == "instruction" and r.get("input"):
                u = u + "\n\n" + r["input"]
            o = r[ko]
            return u, o if isinstance(o, str) else _json.dumps(o, ensure_ascii=False)
    raise ValueError(f"알 수 없는 형식 — 키: {list(r.keys())[:8]}")

pairs = [to_pair(r) for r in raw]
if SMOKE:
    pairs = pairs[:64]
print(f"✅ 샘플 {len(pairs)}건 | 예시 입력 120자: {pairs[0][0][:120]!r}")

In [ ]:
#@title 6️⃣ 모델 로드 (4bit) + LoRA
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

tok = AutoTokenizer.from_pretrained(MODEL_ID)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=P["dtype"], bnb_4bit_use_double_quant=True),
    device_map="auto", torch_dtype=P["dtype"])

model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
model = get_peft_model(model, LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]))
model.print_trainable_parameters()

In [ ]:
#@title 7️⃣ 토크나이즈 (프롬프트 라벨 마스킹)
from datasets import Dataset

def tokenize(ex):
    prompt_ids = tok.apply_chat_template(
        [{"role": "user", "content": ex["u"]}], add_generation_prompt=True)
    out_ids = tok(ex["o"], add_special_tokens=False)["input_ids"] + [tok.eos_token_id]
    ids = (prompt_ids + out_ids)[:P["seqlen"]]
    labels = [-100] * min(len(prompt_ids), len(ids)) + ids[len(prompt_ids):]
    return {"input_ids": ids, "labels": labels}

ds = Dataset.from_list([{"u": u, "o": o} for u, o in pairs]).map(
    tokenize, remove_columns=["u", "o"])
ds = ds.train_test_split(test_size=max(EVAL_RATIO, 4 / len(ds)), seed=42)
print(ds)

In [ ]:
#@title 8️⃣ 학습 — 체크포인트 자동 재개 + 실시간 push
import time as _time
from transformers import Trainer, TrainingArguments, TrainerCallback, DataCollatorForSeq2Seq
from transformers.trainer_utils import get_last_checkpoint

class MissionControlCallback(TrainerCallback):
    def on_log(self, args, state, control, logs=None, **kw):
        if logs and ("loss" in logs or "eval_loss" in logs):
            logger.log({"ts": int(_time.time()), "step": state.global_step,
                        "loss": logs.get("loss"), "eval_loss": logs.get("eval_loss")})

args = TrainingArguments(
    output_dir=CKPT_DIR,
    num_train_epochs=EPOCHS,
    max_steps=20 if SMOKE else -1,
    per_device_train_batch_size=P["bs"],
    gradient_accumulation_steps=P["accum"],
    learning_rate=LR, lr_scheduler_type="cosine", warmup_ratio=0.05,
    bf16=P["bf16"], fp16=P["fp16"],
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    optim="paged_adamw_8bit",
    logging_steps=LOG_EVERY,
    eval_strategy="steps", eval_steps=LOG_EVERY * 5,
    save_strategy="steps", save_steps=SAVE_EVERY, save_total_limit=2,
    report_to="none")

trainer = Trainer(model=model, args=args,
                  train_dataset=ds["train"], eval_dataset=ds["test"],
                  data_collator=DataCollatorForSeq2Seq(tok, padding=True),
                  callbacks=[MissionControlCallback()])

ckpt = get_last_checkpoint(CKPT_DIR)
if not logger.resumed:
    logger.log({"event": "start", "gpu": gpu,
                "model": MODEL_ID + (" [SMOKE]" if SMOKE else ""), "ts": int(_time.time())})
if ckpt:
    print("↻ 체크포인트에서 재개:", ckpt)
try:
    trainer.train(resume_from_checkpoint=ckpt)
finally:
    logger.log({"event": "end", "ts": int(_time.time())})
print("🏁 완료 — 관제센터 FT모니터에서", RUN_ID, "확인")

In [ ]:
#@title 9️⃣ 어댑터 저장 (Drive면 영구 보존)
SAVE = f"{BASE}/adapter"
trainer.model.save_pretrained(SAVE)
tok.save_pretrained(SAVE)
!du -sh {SAVE}
print("✅ 어댑터:", SAVE)
print("   체크포인트:", CKPT_DIR)

---
### 체크리스트
- [ ] T4 + `SMOKE=True` → FT모니터 이력에 `run_001` 등장 (~1 CU)
- [ ] `SMOKE=False`, `RUN_ID` 유지 → **L4** 전체 재실행 (~6–12 CU) — smoke 지표에 이어서 본학습이 누적됨. 깔끔히 새로 시작하려면 `RUN_ID="run_001b"` 등으로 변경
- [ ] (선택) 학습 중 브라우저 닫기 → 백그라운드 실행 확인 / 강제 끊고 재실행 → resume 확인
- [ ] 완료 후: 어댑터 Drive 확인, `ft_logs/run_000_smoke/` 삭제
- 다음: 어댑터 → vLLM 서빙 → `EXTRACT_MODEL` 교체 → 필드 F1 ≥ 90% 검증